In [5]:
import geopandas as gpd
from pathlib import Path

In [6]:
address_gdf = gpd.read_parquet("../../data/processed/nsw_addressing/addresspoint_all.parquet")
bushfire_gdf = gpd.read_parquet("../../data/processed/nsw_bushfire/bushfire.parquet")

In [7]:
print("Address rows:", len(address_gdf))
print("Bushfire rows:", len(bushfire_gdf))

print("Address CRS:", address_gdf.crs)
print("Bushfire CRS:", bushfire_gdf.crs)

print(address_gdf.columns)
print(bushfire_gdf.columns)

Address rows: 4225113
Bushfire rows: 265786
Address CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "dire

In [8]:
bushfire_keep = bushfire_gdf[
    [
        "fid",
        "d_category",
        "d_guidelin",
        "category",
        "guideline",
        "geometry",
    ]
].copy()

if address_gdf.crs != bushfire_keep.crs:
    address_gdf = address_gdf.to_crs(bushfire_keep.crs)

sample_address = address_gdf.sample(50000, random_state=42).copy()
print(len(sample_address))

50000


In [9]:
joined = gpd.sjoin(
    sample_address,
    bushfire_keep,
    how="left",
    predicate="within"
)

print("Joined rows:", len(joined))
joined[
    ["address", "d_category", "d_guidelin"]
].head(10)

Joined rows: 50000


,address,d_category,d_guidelin
2262872,GRINTON AVENUE ASHMONT,NaN,NaN
136524,58 VISTA AVENUE CATALINA,Vegetation Buffer,v5b
3400623,5/40 TRAIN STREET BROULEE,NaN,NaN
975625,13 BACH AVENUE EMERTON,NaN,NaN
1229084,12 GUILFOYLE PLACE CUDGEN,NaN,NaN
3506896,14 SPURFIELD ROAD MCLEANS RIDGES,Vegetation Buffer,v5b
2094798,1/43 THE TRONGATE GRANVILLE,NaN,NaN
1372500,39 MANNS ROAD NARARA,Vegetation Category 1,v5b
466580,4 CAROLINA CRESCENT MUDGEE,NaN,NaN
3991008,62 WADE STREET COOLAMON,NaN,NaN


In [10]:
joined["d_category"].isna().mean()

np.float64(0.82226)

In [11]:
joined["d_category"].value_counts(dropna=False).head(20)

d_category
NaN                      41113
Vegetation Buffer         4679
Vegetation Category 3     2677
Vegetation Category 1     1339
Vegetation Category 2      192
Name: count, dtype: int64

In [12]:
joined["d_category"].value_counts(dropna=False).head(20)

d_category
NaN                      41113
Vegetation Buffer         4679
Vegetation Category 3     2677
Vegetation Category 1     1339
Vegetation Category 2      192
Name: count, dtype: int64

In [13]:
bushfire_hits = joined[joined["d_category"].notna()].copy()
print("Bushfire hits:", len(bushfire_hits))

bushfire_hits[
    ["address", "d_category", "d_guidelin"]
].sample(min(20, len(bushfire_hits)), random_state=42)

Bushfire hits: 8887


,address,d_category,d_guidelin
3708140,1617 OLD COOMA ROAD ROYALLA,Vegetation Category 3,v5b
2126049,56 BENSON ROAD BEAUMONT HILLS,Vegetation Buffer,v5b
3854714,135 OLD BANGALOW ROAD BYRON BAY,Vegetation Category 3,v5b
3314961,281 RACECOURSE ROAD TOTTENHAM,Vegetation Category 3,v5b
2671140,498 MCLELLANS LANE URANGELINE,Vegetation Category 3,v5b
12164,360 BOBUNDARA ROAD BERRIDALE,Vegetation Category 3,v5b
2919097,121 SHEPHERD ROAD SPRING CREEK,Vegetation Category 3,v5b
1529361,362 TOONGA SETTLEMENT ROAD TARCUTTA,Vegetation Category 3,v5b
2595799,65 BUNGA STREET BERMAGUI,Vegetation Buffer,v5b
4125036,542 CURRS ROAD BOLIVIA,Vegetation Category 1,v5b


In [14]:
joined["rid"].duplicated().mean()

np.float64(0.0)

In [15]:
joined["bushfire_flag"] = joined["d_category"].notna().astype(int)
joined["bushfire_flag"].value_counts(normalize=True)

bushfire_flag
0    0.82226
1    0.17774
Name: proportion, dtype: float64

In [16]:
risk_map = {
    "Vegetation Category 1": 3,
    "Vegetation Category 2": 2,
    "Vegetation Category 3": 1,
    "Vegetation Buffer": 1,
}

joined["bushfire_risk_level"] = joined["d_category"].map(risk_map).fillna(0).astype(int)
joined["bushfire_risk_level"].value_counts(dropna=False)

bushfire_risk_level
0    41113
1     7356
3     1339
2      192
Name: count, dtype: int64

In [17]:
Path("../../data/processed/site_features").mkdir(parents=True, exist_ok=True)

joined.to_parquet(
    "../../data/processed/site_features/address_with_bushfire_sample.parquet",
    index=False
)

In [18]:
joined["d_category"].isna().mean()
joined["rid"].duplicated().mean()

np.float64(0.0)

In [19]:
joined["d_category"].value_counts(dropna=False)

d_category
NaN                      41113
Vegetation Buffer         4679
Vegetation Category 3     2677
Vegetation Category 1     1339
Vegetation Category 2      192
Name: count, dtype: int64

In [20]:
joined["rid"].duplicated().mean()

np.float64(0.0)